# Unidad III – Machine Learning con Enfoque en Aplicaciones

## Clase 3 – Desarrollo Orientado a Servicios

* **Curso:** IA y Ciencia de Datos (Certificado – 1 año)
* **Enfoque:** Práctico, orientado a producción
* **Tecnologías base:** Python, FastAPI, Pydantic, Scikit-learn

---


### Objetivo general de la clase

Al finalizar la clase, el estudiante será capaz de:

> **Diseñar, evaluar y mejorar una API de Machine Learning preparada para entornos productivos**, aplicando validaciones estrictas, manejo profesional de errores y principios de arquitectura orientada a servicios.

---


## 1. Desarrollo Orientado a Servicios (Service-Oriented Development)

### 1.1 ¿Qué es desarrollo orientado a servicios?

Basado en **Service-Oriented Architecture (SOA)** y **Microservices**.

#### Definición formal

Un **servicio** es una unidad de software que:

* Expone una **interfaz clara**
* Es **independiente**
* Se comunica mediante protocolos estándar (HTTP/REST)
* Tiene una responsabilidad única

---


### 1.2 API como contrato

Una API **no es código**, es un **contrato**.

Define:

* Qué recibe
* Qué devuelve
* Qué errores puede producir

En ML:

* El modelo **no se expone directamente**
* Se expone **un servicio de inferencia**

---


<img src="img/Cómo-funciona-Restful.png" width="1300" alt="Descripción">

### 2. Buenas prácticas para construir APIs robustas

#### 2.1 Principios clave (industria)

| Principio             | Descripción                     |
| --------------------- | ------------------------------- |
| Single Responsibility | Un endpoint = un propósito      |
| Stateless             | No guarda estado entre requests |
| Validación estricta   | Nunca confiar en el cliente     |
| Errores explícitos    | Nunca fallar silenciosamente    |
| Tipado fuerte         | Reduce errores en producción    |

---


### 2.2 Estructura profesional de una API ML

```
app/
 ├── api/
 │   └── routes.py
 ├── service/
 │   └── model_service.py
 ├── pipeline/
 │   └── pipeline.py
 ├── model/
 │   └── pipeline.joblib
 └── main.py
```

✔ Separación clara de responsabilidades
✔ Facilita pruebas y mantenimiento
✔ Escalable

---


### 2.3 Buenas prácticas específicas para APIs de ML

* No entrenar modelos en endpoints
* Cargar el modelo una sola vez (startup)
* Validar **orden y tipo de features**
* Controlar inputs fuera de rango
* No exponer detalles internos del modelo

---


### 3. Validaciones y manejo de errores profesional

* Datos inválidos
* Errores silenciosos
* Predicciones absurdas

---


#### 3.1 Manejo correcto de errores

* Mensaje claro
* Código correcto
* Sin filtrar detalles internos

---


#Enunciado del Proyecto – Módulo 3 (Data Science & AI)

## **Objetivo**

Construir una aplicación de Machine Learning completa y profesional, que entrene un modelo con datos reales y exponga predicciones mediante una API REST con FastAPI.

### **Contexto**

Trabajarás con un dataset real de consumo energético doméstico para predecir el consumo de electrodomésticos (Wh). El enfoque será de regresión y deberás aplicar buenas prácticas de ingeniería de software, ML en producción y validaciones estrictas de entrada.

### **Requisitos obligatorios**

#### **Dataset real desde internet**
* Descargar desde URL pública.
* Cachear localmente.
#### **Pipeline profesional (Scikit-learn)**
* SimpleImputer + StandardScaler + ColumnTransformer
* Modelo: ElasticNet
#### **Entrenamiento y evaluación**
* Train/test split
* Métricas: MAE, MSE, RMSE, R²
* Guardar el pipeline con joblib
#### **API REST profesional (FastAPI)**
* Endpoint POST /predict
* Validaciones estrictas con Pydantic
* Manejo de errores 400, 503, 500
* Cargar modelo en el arranque, no entrenar en endpoints
#### **Documentación y buenas prácticas**
* README completo
* PEP8, tipado, docstrings
* Arquitectura por capas (API, Service, Pipeline, Training)
#### **Entregables**
* Código funcional en la estructura indicada.
* README con instrucciones claras.
* Guía de pruebas de API.
* Evidencia de entrenamiento y métricas.

#### **Arquitectura del proyecto**

```
project/
│
├── app/
│   │
│   ├── main.py 
│   │
│   ├── api/
│   │   └── routes.py
│   ├── service/
│   │   └── model_service.py
│
├── data/
│
├── pipeline/
│   └── pipeline.py
│
├── training/
│   └── train.py
│
├── model/
│   └── pipeline.joblib
│
├── requirements.txt
└── README.md
```


`pipeline/pipeline.py`: datos, columnas, caché, y construcción del pipeline (base conceptual de ML).

In [ ]:
"""Funciones para cargar datos y construir el pipeline de ML."""  # Docstring

from __future__ import annotations  # Habilita anotaciones futuras

from pathlib import Path  # Manejo de rutas
from typing import List, Tuple  # Tipos de retorno
from urllib.request import urlretrieve  # Descarga por URL

import pandas as pd  # Manejo de datos tabulares
from sklearn.compose import ColumnTransformer  # Preprocesamiento por columnas
from sklearn.impute import SimpleImputer  # Imputación de nulos
from sklearn.linear_model import ElasticNet  # Modelo de regresión
from sklearn.pipeline import Pipeline  # Pipeline de ML
from sklearn.preprocessing import StandardScaler  # Escalado estándar


def get_dataset_url() -> str:  # URL del dataset
    """Retorna la URL pública del dataset real."""  # Docstring
    return (  # Construye URL
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"  # Base
        "00374/energydata_complete.csv"  # Archivo
    )  # Fin de URL


def get_cache_path() -> Path:  # Ruta de caché
    """Retorna la ruta local de caché del dataset."""  # Docstring
    return Path("data") / "energydata_complete.csv"  # Path local


def get_feature_columns() -> List[str]:  # Columnas explicativas
    """Retorna las columnas explicativas usadas para el modelo."""  # Docstring
    return ["T1", "RH_1", "T_out", "Windspeed"]  # Lista de features


def get_target_column() -> str:  # Columna objetivo
    """Retorna la columna objetivo del problema."""  # Docstring
    return "Appliances"  # Target del dataset


def download_dataset(url: str, cache_path: Path) -> Path:  # Descarga con caché
    """Descarga el dataset desde una URL pública si no existe en caché."""  # Docstring
    cache_path.parent.mkdir(parents=True, exist_ok=True)  # Asegura carpeta
    if cache_path.exists():  # Si ya existe
        return cache_path  # Devuelve ruta actual
    urlretrieve(url, cache_path.as_posix())  # Descarga dataset
    return cache_path  # Retorna ruta


def load_dataset() -> pd.DataFrame:  # Carga el dataset
    """Carga el dataset desde caché, descargándolo si es necesario."""  # Docstring
    dataset_path = download_dataset(get_dataset_url(), get_cache_path())  # Ruta final
    df = pd.read_csv(dataset_path)  # Lee CSV
    columns = get_feature_columns() + [get_target_column()]  # Selección de columnas
    return df[columns].copy()  # Retorna copia filtrada


def build_pipeline(feature_columns: List[str]) -> Pipeline:  # Construye pipeline
    """Construye el pipeline de preprocesamiento y modelo."""  # Docstring
    numeric_transformer = Pipeline(  # Pipeline numérico
        steps=[  # Lista de pasos
            ("imputer", SimpleImputer(strategy="median")),  # Imputa medianas
            ("scaler", StandardScaler()),  # Estandariza
        ]  # Fin de pasos
    )  # Fin del pipeline
    preprocessor = ColumnTransformer(  # Preprocesador por columnas
        transformers=[("num", numeric_transformer, feature_columns)]  # Mapea cols
    )  # Fin del preprocesador
    model = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42)  # Modelo
    return Pipeline(steps=[("preprocess", preprocessor), ("model", model)])  # Final


def split_features_target(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:  # Split
    """Separa variables explicativas y objetivo."""  # Docstring
    feature_columns = get_feature_columns()  # Lista de features
    target_column = get_target_column()  # Nombre de target
    return df[feature_columns], df[target_column]  # Devuelve X, y


`training/train.py:` entrenamiento, métricas y guardado del modelo (flujo ML completo).

In [ ]:
"""Entrenamiento y evaluación del modelo de consumo energético."""  # Docstring

from __future__ import annotations  # Habilita anotaciones futuras

from math import sqrt  # Raíz cuadrada para RMSE
from pathlib import Path  # Manejo de rutas
import joblib  # Guardado de modelos
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Métricas
from sklearn.model_selection import train_test_split  # Separación train/test

from pipeline.pipeline import (  # Utilidades del pipeline
    build_pipeline,  # Constructor del pipeline
    get_feature_columns,  # Columnas explicativas
    load_dataset,  # Carga de dataset
    split_features_target,  # Separación X/y
)


def train_and_evaluate(model_path: Path) -> None:  # Entrena y evalúa
    """Entrena el pipeline, evalúa y guarda el modelo entrenado."""  # Docstring
    df = load_dataset()  # Carga dataset
    features, target = split_features_target(df)  # Separa X/y

    x_train, x_test, y_train, y_test = train_test_split(  # Split train/test
        features, target, test_size=0.2, random_state=42  # Proporción y semilla
    )

    pipeline = build_pipeline(get_feature_columns())  # Construye pipeline
    pipeline.fit(x_train, y_train)  # Entrena modelo

    predictions = pipeline.predict(x_test)  # Predice en test
    mae = mean_absolute_error(y_test, predictions)  # Calcula MAE
    mse = mean_squared_error(y_test, predictions)  # Calcula MSE
    rmse = _compute_rmse(y_test, predictions, mse)  # Calcula RMSE
    r2 = r2_score(y_test, predictions)  # Calcula R2

    print("Métricas de evaluación en conjunto de prueba (test):")  # Encabezado
    print(f"- MAE  (Error absoluto medio)         : {mae:.4f}  (menor es mejor)")  # MAE
    print(f"- MSE  (Error cuadrático medio)       : {mse:.4f}  (menor es mejor)")  # MSE
    print(f"- RMSE (Raíz del MSE)                 : {rmse:.4f}  (menor es mejor)")  # RMSE
    print(  # R2 con formato
        f"- R2   (Coeficiente de determinación): {r2:.4f}  (más cerca a 1 es mejor)"
    )

    model_path.parent.mkdir(parents=True, exist_ok=True)  # Crea carpeta si falta
    joblib.dump(pipeline, model_path)  # Guarda el pipeline
    print(f"Pipeline guardado en: {model_path}")  # Mensaje final


def _compute_rmse(  # Helper de RMSE
    y_true, y_pred, mse_value: float  # Datos y MSE precalculado
) -> float:  # Retorno float
    """Calcula RMSE con compatibilidad entre versiones de scikit-learn."""  # Docstring
    try:  # Intenta métrica nueva
        from sklearn.metrics import root_mean_squared_error  # Métrica moderna

        return float(root_mean_squared_error(y_true, y_pred))  # Retorna RMSE
    except Exception:  # Fallback genérico
        return float(sqrt(mse_value))  # Calcula RMSE manual


def main() -> None:  # Entry point
    """Punto de entrada del entrenamiento."""  # Docstring
    model_path = Path("model") / "pipeline.joblib"  # Ruta del modelo
    train_and_evaluate(model_path)  # Ejecuta entrenamiento


if __name__ == "__main__":  # Ejecución directa
    main()  # Llama a main


`app/service/model_service.py:` carga del modelo y lógica de inferencia (capa de servicio).

In [ ]:
"""Capa de servicio para el modelo de predicción."""  # Docstring del módulo

from __future__ import annotations  # Habilita anotaciones futuras

from dataclasses import dataclass  # Decorador para clases simples
from pathlib import Path  # Manejo de rutas
from typing import Dict  # Tipado de diccionarios

import joblib  # Serialización de modelos
import pandas as pd  # DataFrames para inferencia
from sklearn.pipeline import Pipeline  # Tipo del pipeline

from pipeline.pipeline import get_feature_columns  # Columnas esperadas


class ModelNotLoadedError(RuntimeError):  # Error personalizado
    """Error cuando el modelo no está disponible."""  # Docstring


@dataclass  # Genera __init__ y utilidades
class ModelService:  # Servicio de inferencia
    """Servicio de predicción que encapsula el pipeline entrenado."""  # Docstring

    model_path: Path  # Ruta del modelo
    pipeline: Pipeline  # Pipeline entrenado

    @classmethod  # Método de clase
    def load(cls, model_path: Path) -> "ModelService":  # Carga servicio
        """Carga el pipeline desde disco."""  # Docstring
        if not model_path.exists():  # Valida existencia
            raise ModelNotLoadedError("Modelo no disponible.")  # Error 503
        try:  # Manejo seguro de lectura
            pipeline = joblib.load(model_path)  # Carga el pipeline
        except Exception as exc:  # pragma: no cover - protección defensiva
            raise ModelNotLoadedError("Modelo no disponible.") from exc  # Envuelve error
        return cls(model_path=model_path, pipeline=pipeline)  # Instancia servicio

    def predict(self, features: Dict[str, float]) -> float:  # Predice un valor
        """Genera una predicción única a partir de las features."""  # Docstring
        feature_columns = get_feature_columns()  # Orden esperado
        data = pd.DataFrame([features], columns=feature_columns)  # DataFrame 1xN
        prediction = self.pipeline.predict(data)[0]  # Predicción escalar
        return float(prediction)  # Conversión a float nativo


`app/api/routes.py:` contrato REST, validaciones y endpoint /predict (capa API).

In [ ]:
"""Rutas de la API para predicción."""  # Docstring del módulo

from __future__ import annotations  # Habilita anotaciones futuras

from typing import Any  # Tipado general de retorno

from fastapi import APIRouter, Request, status  # Componentes FastAPI
from pydantic import BaseModel, Field, ConfigDict  # Validación y schemas

from app.service.model_service import (  # Servicio y error de dominio
    ModelNotLoadedError,
    ModelService,
)

router = APIRouter()  # Enrutador de la API


class PredictionRequest(BaseModel):  # Modelo de entrada
    """Esquema de entrada para predicciones."""  # Docstring de clase

    model_config = ConfigDict(  # Configuración de validación
        extra="forbid",  # Rechaza campos no definidos
        json_schema_extra={  # Metadatos para Swagger
            "examples": [  # Lista de ejemplos
                {"T1": 21.0, "RH_1": 45.0, "T_out": 10.0, "Windspeed": 3.5}  # Ejemplo
            ]  # Fin de ejemplos
        },  # Fin de metadata
    )  # Fin de config

    T1: float = Field(  # Temperatura interior
        ..., ge=0.0, le=50.0, description="Temperatura en cocina (C)"
    )
    RH_1: float = Field(  # Humedad interior
        ..., ge=0.0, le=100.0, description="Humedad en cocina (%)"
    )
    T_out: float = Field(  # Temperatura exterior
        ..., ge=-30.0, le=50.0, description="Temperatura exterior (C)"
    )
    Windspeed: float = Field(  # Velocidad del viento
        ..., ge=0.0, le=20.0, description="Velocidad del viento (m/s)"
    )


class PredictionResponse(BaseModel):  # Modelo de salida
    """Esquema de respuesta de predicción."""  # Docstring de clase

    model_config = ConfigDict(  # Configuración del schema
        json_schema_extra={  # Metadatos para Swagger
            "examples": [  # Lista de ejemplos
                {  # Ejemplo de respuesta
                    "prediction": 120.45,  # Valor estimado
                    "model": "ElasticNet v1.0",  # Modelo usado
                    "unit": "energy consumption (Wh)",  # Unidad
                }
            ]  # Fin de ejemplos
        }  # Fin de metadata
    )  # Fin de config

    prediction: float  # Predicción numérica
    model: str  # Identificador de modelo
    unit: str  # Unidad de salida


def get_service(request: Request) -> ModelService:  # Obtiene servicio
    """Recupera el servicio desde el estado de la app."""  # Docstring
    service = request.app.state.model_service  # Servicio cargado
    if service is None:  # Valida disponibilidad
        raise ModelNotLoadedError("Modelo no cargado.")  # Error 503
    return service  # Devuelve servicio


@router.post(  # Declaración del endpoint
    "/predict",  # Ruta
    response_model=PredictionResponse,  # Modelo de respuesta
    status_code=status.HTTP_200_OK,  # Código HTTP
    summary="Predice consumo energético",  # Resumen Swagger
    response_description="Predicción del consumo en Wh",  # Descripción Swagger
    tags=["predictions"],  # Tag de agrupación
)
def predict(request: PredictionRequest, http_request: Request) -> Any:  # Endpoint
    """Endpoint de predicción del consumo energético."""  # Docstring
    service = get_service(http_request)  # Obtiene servicio
    payload = request.model_dump()  # Serializa entrada
    prediction = service.predict(payload)  # Ejecuta inferencia
    return PredictionResponse(  # Construye respuesta
        prediction=prediction,  # Valor estimado
        model="ElasticNet v1.0",  # Modelo reportado
        unit="energy consumption (Wh)",  # Unidad
    )


`app/main.py:` creación de la app, manejo de errores y arranque de Uvicorn (bootstrap del backend).

In [ ]:
"""Aplicación FastAPI para predicción de consumo energético."""  # Docstring

from __future__ import annotations  # Habilita anotaciones futuras

import logging  # Logging estándar
import sys  # Acceso a sys.path
from contextlib import asynccontextmanager  # Lifespan moderno
from pathlib import Path  # Manejo de rutas
import threading  # Hilo para abrir navegador
import time  # Espera antes de abrir Swagger
import webbrowser  # Abre navegador

from fastapi import FastAPI, Request, status  # Componentes FastAPI
from fastapi.exceptions import RequestValidationError  # Errores de validación
from fastapi.responses import JSONResponse  # Respuestas JSON
import uvicorn  # Servidor ASGI

ROOT_DIR = Path(__file__).resolve().parents[1]  # Raíz del proyecto
if str(ROOT_DIR) not in sys.path:  # Verifica sys.path
    sys.path.insert(0, str(ROOT_DIR))  # Agrega raíz al path

from app.api.routes import router  # Importa rutas
from app.service.model_service import ModelNotLoadedError, ModelService  # Servicio y error


logging.basicConfig(level=logging.INFO)  # Configura logging básico
logger = logging.getLogger(__name__)  # Logger del módulo


@asynccontextmanager  # Decorador de lifespan
async def lifespan(app: FastAPI):  # Ciclo de vida de la app
    """Manejo de eventos de ciclo de vida."""  # Docstring
    model_path = Path("model") / "pipeline.joblib"  # Ruta del modelo
    try:  # Intenta cargar el modelo
        app.state.model_service = ModelService.load(model_path)  # Carga servicio
    except ModelNotLoadedError:  # Si falla la carga
        app.state.model_service = None  # Marca servicio vacío
        logger.error("Modelo no disponible en startup.")  # Registra error
    yield  # Continúa con la vida de la app


def create_app() -> FastAPI:  # Factory de app
    """Crea y configura la aplicación FastAPI."""  # Docstring
    app = FastAPI(  # Instancia de la app
        title="Energy Consumption Prediction API",  # Título
        version="1.0.0",  # Versión
        lifespan=lifespan,  # Lifespan configurado
    )
    app.include_router(router)  # Registra rutas

    @app.exception_handler(RequestValidationError)  # Handler de 400
    def validation_exception_handler(  # Función handler
        _request: Request, exc: RequestValidationError  # Request y error
    ) -> JSONResponse:  # Tipo de retorno
        return JSONResponse(  # Respuesta JSON
            status_code=status.HTTP_400_BAD_REQUEST,  # Código 400
            content={"error": "Datos inválidos.", "details": exc.errors()},  # Payload
        )

    @app.exception_handler(ModelNotLoadedError)  # Handler de 503
    def model_not_loaded_handler(  # Función handler
        _request: Request, _exc: ModelNotLoadedError  # Request y error
    ) -> JSONResponse:  # Tipo de retorno
        return JSONResponse(  # Respuesta JSON
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,  # Código 503
            content={"error": "Modelo no cargado."},  # Payload
        )

    @app.exception_handler(Exception)  # Handler genérico
    def generic_exception_handler(  # Función handler
        _request: Request, exc: Exception  # Request y excepción
    ) -> JSONResponse:  # Tipo de retorno
        logger.exception("Error interno no controlado: %s", exc)  # Log del error
        return JSONResponse(  # Respuesta JSON
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,  # Código 500
            content={"error": "Error interno."},  # Payload
        )

    return app  # Devuelve la app


app = create_app()  # Instancia global de la app


def main() -> None:  # Entry point
    """Ejecuta la app con Uvicorn para pruebas locales."""  # Docstring
    def _open_docs() -> None:  # Función interna
        time.sleep(1.5)  # Espera por el arranque
        webbrowser.open_new_tab("http://localhost:8000/docs")  # Abre Swagger

    threading.Thread(target=_open_docs, daemon=True).start()  # Lanza hilo
    uvicorn.run(  # Ejecuta servidor
        app,  # App ASGI
        host="127.0.0.1",  # Host local
        port=8000,  # Puerto
        reload=False,  # Sin recarga
    )


if __name__ == "__main__":  # Ejecución directa
    main()  # Llama a main


### **Para ejecuTar la app, leer el archivo TESTING_GUIDE.PDF